# 랜덤신호론 12주차 실습
## 머신러닝 학습 데이터 만들기

1. 원신호와 특징값을 구분한다.
2. 여러 신호에서 시간·주파수 특징을 추출한다.
3. 입력 X와 정답 y를 구성한다.
4. 훈련·시험 분리와 데이터 누수를 설명한다.


## 0. 공통 환경 준비


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

rng=np.random.default_rng(2026)
plt.style.use('seaborn-v0_8-whitegrid')
FEATURE_NAMES=['mean','std','peak_to_peak','rms','autocorrelation','dominant_frequency','spectral_energy']

def lag1_autocorrelation(x):
    x=np.asarray(x,float)
    if len(x)<2 or np.std(x[:-1])==0 or np.std(x[1:])==0: return 0.0
    return float(np.corrcoef(x[:-1],x[1:])[0,1])

def extract_features(x,fs):
    x=np.asarray(x,float); centered=x-x.mean()
    spectrum=np.abs(np.fft.rfft(centered)); freqs=np.fft.rfftfreq(len(x),1/fs)
    return {'mean':float(x.mean()),'std':float(x.std()),'peak_to_peak':float(np.ptp(x)),
      'rms':float(np.sqrt(np.mean(x**2))),'autocorrelation':lag1_autocorrelation(x),
      'dominant_frequency':float(freqs[np.argmax(spectrum)]),
      'spectral_energy':float(np.sum(spectrum**2)/len(x))}

def generate_signal(kind,fs=50,seconds=4,seed=0):
    local=np.random.default_rng(seed); t=np.arange(0,seconds,1/fs)
    clean=np.sin(2*np.pi*3*t)+.35*np.sin(2*np.pi*7*t)
    if kind=='normal': x=clean+local.normal(0,.18,len(t))
    elif kind=='high_noise': x=clean+local.normal(0,.75,len(t))
    elif kind=='spike':
        x=clean+local.normal(0,.18,len(t)); idx=local.choice(len(t),4,False); x[idx]+=local.choice([-5,5],4)
    else: raise ValueError('kind 확인')
    return t,x

def build_dataset(n=40,fs=50):
    rows=[]; labels=['normal','high_noise','spike']
    for j,label in enumerate(labels):
        for i in range(n):
            _,x=generate_signal(label,fs=fs,seed=1000+100*j+i)
            row=extract_features(x,fs); row['label']=label; rows.append(row)
    return pd.DataFrame(rows)
print('준비 완료')


## 1. 신호 한 개를 특징 한 행으로 변환


In [ ]:
fs=50;_,sig=generate_signal('normal',fs,seed=1)
print(pd.DataFrame([extract_features(sig,fs)]));print('원신호 표본:',len(sig),'특징:',len(FEATURE_NAMES))


**관찰 1** 원신호 표본 수와 특징 수를 비교하고 요약의 장단점을 작성하세요.


## 2. 특징표 만들기


In [ ]:
df=build_dataset(40,fs);print(df.head());print(df.shape);print(df.label.value_counts())


## 3. 특징 분포


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
for label,g in df.groupby('label'):
    ax[0].hist(g['std'],alpha=.5,label=label);ax[1].hist(g['peak_to_peak'],alpha=.5,label=label)
ax[0].legend();plt.show()


## 4. 훈련·시험 분리


In [ ]:
X=df[FEATURE_NAMES];y=df.label
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
print(X_train.shape,X_test.shape);print(y_train.value_counts(normalize=True).round(2));print(y_test.value_counts(normalize=True).round(2))


**미니 과제** 훈련과 시험의 역할, 데이터 누수의 문제를 설명하세요.


## 참고 자료
- train_test_split: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- Common pitfalls: https://scikit-learn.org/stable/common_pitfalls.html
